In [1]:
! pip install transformers accelerate torch

Below code is also for generating instructions but for qwen 2.5_3B

In [7]:
import json
import os
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# ============================================================
# Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

INPUT_PATH = "/content/seed_sample_combined.jsonl"
OUTPUT_PATH = "/content/instructions_sample_Qwen2.5_3B.jsonl"


# ============================================================
# Check GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required. "
        "In Colab, go to Runtime -> Change runtime type -> GPU."
    )

device = "cuda"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


# ============================================================
# Load tokenizer
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

print("Tokenizer loaded.")


# ============================================================
# Load model
# ============================================================

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully.")


# ============================================================
# Prompt
# ============================================================

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


# ============================================================
# Generate instruction
# ============================================================

def generate_instruction(
    doc: str,
    max_new_tokens: int = 200
) -> str:

    messages = [
        {
            "role": "user",
            "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc),
        }
    ]

    # Qwen2.5 supports the standard Transformers chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    ).to(model.device)

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=max_new_tokens,

            # Sampling parameters
            do_sample=True,
            temperature=0.7,
            top_p=0.9,

            # Avoid unnecessary repetition
            repetition_penalty=1.05,

            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove input tokens from generated output
    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    generated = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated


# ============================================================
# Resume support
# ============================================================

already_done = set()

if os.path.exists(OUTPUT_PATH):

    with open(
        OUTPUT_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                result = json.loads(line)

                if "id" in result:
                    already_done.add(result["id"])

            except json.JSONDecodeError:
                pass

    print(
        f"Resuming: {len(already_done)} already processed"
    )


# ============================================================
# Count total input items
# ============================================================

with open(
    INPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    total_items = sum(
        1
        for line in f
        if line.strip()
    )

print(
    f"Total input items: {total_items}"
)


# ============================================================
# Process input JSONL
# ============================================================

if __name__ == "__main__":

    with open(
        INPUT_PATH,
        "r",
        encoding="utf-8"
    ) as infile, \
    open(
        OUTPUT_PATH,
        "a",
        encoding="utf-8"
    ) as outfile:

        for line in tqdm(
            infile,
            total=total_items,
            desc="Generating instructions"
        ):

            if not line.strip():
                continue

            item = json.loads(line)

            item_id = item["id"]

            # Skip already processed items
            if item_id in already_done:
                continue

            doc = item["text"]

            try:

                instruction = generate_instruction(
                    doc,
                    max_new_tokens=200
                )

            except Exception as e:

                print(
                    f"\nFailed on id={item_id}: {e}"
                )

                continue

            result = {
                "id": item_id,
                "source": item.get(
                    "source",
                    "unknown"
                ),
                "seed_doc": doc,
                "instruction": instruction,
            }

            outfile.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                ) + "\n"
            )

            # Save immediately so interrupted runs can resume
            outfile.flush()

            already_done.add(item_id)


print("Done!")
print(
    f"Output saved to: {OUTPUT_PATH}"
)

GPU: Tesla T4
CUDA: 13.0
Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (6 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Total input items: 150


Generating instructions: 100%|██████████| 150/150 [07:33<00:00,  3.02s/it]

Done!
Output saved to: /content/instructions_sample_Qwen2.5_3B.jsonl


Below code is for evaluating the generated instructions using LLM as a judge

In [1]:
import json
import os
import torch
from pydantic import BaseModel, Field
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Selected Judge Model from EuroEval top rankers
#JUDGE_MODEL_ID = "mistralai/Mistral-Small-24B-Instruct-2501"
JUDGE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
INPUT_PATH = "/content/instructions_sample_Qwen3.5_4B.jsonl"
OUTPUT_PATH = "evaluation_results_qwen3.5_4B.jsonl"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

JUDGE_PROMPT_TEMPLATE = """Du är en expertutvärderare för svensk språkdata och instruktionsfinstämning.
Din uppgift är att bedöma kvaliteten på en AI-genererad instruktion baserad på en givna källtext (seed document).

---
[KÄLLTEXT / SEED DOC]:
{seed_doc}

[GENERERAD INSTRUKTION]:
{instruction}
---

Utvärdera följande två kriterier på en skala från 1 till 5:

1. **relevance_score** (1-5): Hur väl stämmer instruktionen överens med källtexten? Skulle källtexten utgöra ett naturligt och korrekt svar på instruktionen?
2. **idiomaticity_score** (1-5): Hur naturlig, korrekta och idiomatisk är den svenska språknivån i instruktionen? (Inget direktöversatt eller konstlat språk).

Svara EXAKT i följande JSON-format och inget annat:
{{
  "relevance_score": <heltal 1-5>,
  "idiomaticity_score": <heltal 1-5>,
  "reasoning": "<kort motivering på svenska>"
}}"""


def judge_pair(seed_doc: str, instruction: str) -> dict:
    prompt = JUDGE_PROMPT_TEMPLATE.format(seed_doc=seed_doc, instruction=instruction)
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,  # Greedy decoding for consistent LLM-as-a-judge evaluation
            temperature=0.0,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    response_text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Parse judge output
    try:
        # Extract JSON blob if wrap in markdown code fence
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()

        parsed = json.loads(response_text)
        return parsed
    except Exception as e:
        return {
            "relevance_score": None,
            "idiomaticity_score": None,
            "reasoning": f"Failed to parse model response: {response_text}",
        }


def main():
    # Load processed records
    records = []
    if not os.path.exists(INPUT_PATH):
        print(f"Error: {INPUT_PATH} does not exist.")
        return

    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line.strip()))

    # Support resume logic
    already_evaluated = set()
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                already_evaluated.add(json.loads(line)["id"])
        print(f"Resuming: {len(already_evaluated)} samples already evaluated.")

    # Evaluate each generated instruction pair
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        for item in tqdm(records, desc="Evaluating with LLM Judge"):
            item_id = item["id"]
            if item_id in already_evaluated:
                continue

            evaluation = judge_pair(
                seed_doc=item["seed_doc"], instruction=item["instruction"]
            )

            result = {
                "id": item_id,
                "seed_doc": item["seed_doc"],
                "instruction": item["instruction"],
                "judge_model": JUDGE_MODEL_ID,
                "eval_relevance": evaluation.get("relevance_score"),
                "eval_idiomaticity": evaluation.get("idiomaticity_score"),
                "eval_reasoning": evaluation.get("reasoning"),
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()


if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Evaluating with LLM Judge: 100%|██████████| 138/138 [18:00<00:00,  7.83s/it]
